# W2 D2 - RCA Pipeline

Pipeline: load W2-D1 clusters -> graph + temporal scoring -> incident retrieval -> kNN-style class/action selection -> `results/rca_output.json`.

In [1]:
import json
from collections import defaultdict, deque
from datetime import datetime
from pathlib import Path

try:
    import networkx as nx
except ImportError:
    nx = None

BASE = Path.cwd()
CLUSTERS_PATH = BASE / 'results' / 'cluster_summary.json'
ALERTS_PATH = BASE / 'dataset' / 'alerts_sample.jsonl'
SERVICES_PATH = BASE / 'dataset' / 'services.json'
HISTORY_PATH = BASE / 'dataset' / 'incidents_history.json'
OUT_PATH = BASE / 'results' / 'rca_output.json'

SEVERITY_RANK = {'info': 0, 'warn': 1, 'warning': 1, 'crit': 2, 'critical': 2}


def parse_ts(value):
    return datetime.fromisoformat(value.replace('Z', '+00:00'))


def load_json(path):
    return json.loads(path.read_text(encoding='utf-8'))


def load_alerts(path):
    alerts = []
    with path.open(encoding='utf-8') as f:
        for line in f:
            if not line.strip():
                continue
            alert = json.loads(line)
            alert['timestamp'] = parse_ts(alert.get('timestamp') or alert['ts'])
            alert['fingerprint'] = f"{alert['service']}|{alert['metric']}|{alert['severity']}"
            alerts.append(alert)
    return alerts

clusters_doc = load_json(CLUSTERS_PATH)
alerts = load_alerts(ALERTS_PATH)
services_doc = load_json(SERVICES_PATH)
history = load_json(HISTORY_PATH)

print(f"clusters={clusters_doc['output_clusters']} alerts={len(alerts)} history={len(history)}")
print(f"networkx_available={nx is not None}")

clusters=3 alerts=20 history=30
networkx_available=False


In [2]:
def build_graph(services_doc):
    graph = defaultdict(set)
    reverse_graph = defaultdict(set)
    stores = {store['name'] for store in services_doc.get('stores', [])}
    nodes = set(stores)

    for service in services_doc.get('services', []):
        nodes.add(service['name'] if isinstance(service, dict) else service)
    for node in nodes:
        graph[node]
        reverse_graph[node]

    for edge in services_doc['edges']:
        src, dst = (edge['from'], edge['to']) if isinstance(edge, dict) else edge
        graph[src].add(dst)
        reverse_graph[dst].add(src)
        graph[dst]
        reverse_graph[src]
    return graph, reverse_graph, stores


def shortest_distance(graph, start, target):
    if start == target:
        return 0
    q = deque([(start, 0)])
    seen = {start}
    while q:
        node, dist = q.popleft()
        for nxt in graph[node]:
            if nxt == target:
                return dist + 1
            if nxt not in seen:
                seen.add(nxt)
                q.append((nxt, dist + 1))
    return None


def reachable_count(reverse_graph, candidate, services):
    # Upstream services that can be affected if candidate fails.
    q = deque([candidate])
    seen = {candidate}
    while q:
        node = q.popleft()
        for upstream in reverse_graph[node]:
            if upstream not in seen:
                seen.add(upstream)
                q.append(upstream)
    return len(seen & set(services))

graph, reverse_graph, stores = build_graph(services_doc)
edge_count = sum(len(v) for v in graph.values())
subgraph_nodes = set(clusters_doc['clusters'][0]['services'])
print(f"graph_nodes={len(graph)} directed_edges={edge_count}")
print(f"stores={sorted(stores)}")
print(f"subgraph={sorted(subgraph_nodes)}")

graph_nodes=14 directed_edges=17
stores=['cart-redis', 'catalog-db', 'kafka-events', 'payments-db']
subgraph=['cart-svc', 'checkout-svc', 'edge-lb', 'notification-svc', 'payment-svc']


In [3]:
def cluster_alerts(cluster, alerts):
    fingerprints = set(cluster['fingerprints'])
    services = set(cluster['services'])
    start, end = [parse_ts(ts) for ts in cluster['time_range']]
    return [
        alert for alert in alerts
        if alert['service'] in services
        and alert['fingerprint'] in fingerprints
        and start <= alert['timestamp'] <= end
    ]


def first_alert_by_service(cluster, alerts):
    first = {}
    for alert in cluster_alerts(cluster, alerts):
        svc = alert['service']
        if svc not in first or alert['timestamp'] < first[svc]:
            first[svc] = alert['timestamp']
    return first


def graph_score(service, cluster_services):
    services = set(cluster_services)
    impact = reachable_count(reverse_graph, service, services) / max(len(services), 1)
    entry_distances = [shortest_distance(graph, 'edge-lb', service)]
    entry_distances = [dist for dist in entry_distances if dist is not None]
    depth = min(max(entry_distances) / 3, 1.0) if entry_distances else 0.2
    store_penalty = 0.85 if service in stores else 1.0
    return min(1.0, store_penalty * (0.7 * impact + 0.3 * depth))


def timestamp_score(service, first_by_service):
    if not first_by_service:
        return 0.0
    times = list(first_by_service.values())
    start, end = min(times), max(times)
    span = max((end - start).total_seconds(), 1)
    if service not in first_by_service:
        return 0.0
    offset = (first_by_service[service] - start).total_seconds()
    return 1 - (offset / span)


def rank_cluster(cluster):
    first = first_alert_by_service(cluster, alerts)
    rows = []
    for service in cluster['services']:
        g = graph_score(service, cluster['services'])
        t = timestamp_score(service, first)
        final = round(0.6 * g + 0.4 * t, 2)
        rows.append((service, final, round(g, 2), round(t, 2)))
    rows.sort(key=lambda row: row[1], reverse=True)
    return rows

for cluster in clusters_doc['clusters']:
    top3 = [[svc, score] for svc, score, _, _ in rank_cluster(cluster)[:3]]
    print(cluster['cluster_id'], top3)

c-001-000 [['payment-svc', 0.77], ['checkout-svc', 0.47], ['cart-svc', 0.44]]
c-001-001 [['recommender-svc', 0.94]]
c-001-002 [['search-svc', 0.88]]


In [4]:
def incident_similarity(cluster, incident):
    cluster_services = set(cluster['services'])
    incident_services = set(incident.get('services', []))
    cluster_fps = set(cluster.get('fingerprints', []))
    incident_fps = set(incident.get('fingerprints', []))

    root_bonus = 0.3 if incident.get('root_cause_service') in cluster_services else 0.0
    service_union = cluster_services | incident_services
    service_overlap = len(cluster_services & incident_services) / max(len(service_union), 1)
    severity_bonus = 0.15 if incident.get('severity') == cluster.get('max_severity') else 0.0
    fp_union = cluster_fps | incident_fps
    fp_overlap = len(cluster_fps & incident_fps) / max(len(fp_union), 1)
    return round(root_bonus + 0.35 * service_overlap + severity_bonus + 0.2 * fp_overlap, 3)


def top_k_similar(cluster, k=3):
    scored = [(incident_similarity(cluster, incident), incident) for incident in history]
    scored.sort(key=lambda item: item[0], reverse=True)
    return scored[:k]


def classify_from_history(cluster):
    similar = top_k_similar(cluster, 3)
    if not similar or similar[0][0] <= 0:
        return {
            'class': 'other',
            'actions': ['Investigate manually'],
            'confidence': 0.2,
            'similar_incidents': [],
            'method': 'graph+retrieval-fallback',
        }
    confidence = min(0.95, round(0.55 + similar[0][0] * 0.45, 2))
    best = similar[0][1]
    return {
        'class': best['root_cause_class'],
        'actions': best.get('actions', ['Investigate manually']),
        'confidence': confidence,
        'similar_incidents': [incident['incident_id'] for _, incident in similar],
        'method': 'graph+retrieval',
    }

for cluster in clusters_doc['clusters']:
    print(cluster['cluster_id'], [(score, incident['incident_id']) for score, incident in top_k_similar(cluster)])

c-001-000 [(0.77, 'INC-2026-05-10'), (0.77, 'INC-2026-06-01'), (0.757, 'INC-2026-03-22')]
c-001-001 [(1.0, 'INC-2026-02-02'), (1.0, 'INC-2026-06-05'), (0.825, 'INC-2025-12-02')]
c-001-002 [(1.0, 'INC-2026-03-03'), (0.825, 'INC-2026-04-18'), (0.617, 'INC-2025-07-19')]


In [5]:
def reasoning_for(cluster, top_service):
    if cluster['alert_count'] == 1:
        return f"{top_service} is the only alerted service in this cluster, so the RCA remains local unless more topology evidence appears."
    return f"{top_service} is ranked highest because it is downstream in the dependency graph, can explain upstream/victim alerts, and its alert appears early in the cluster time range."

results = []
for cluster in clusters_doc['clusters']:
    ranked = rank_cluster(cluster)
    graph_top3 = [[svc, score] for svc, score, _, _ in ranked[:3]]
    root_cause = graph_top3[0][0]
    classification = classify_from_history(cluster)
    results.append({
        'cluster_id': cluster['cluster_id'],
        'graph_top3': graph_top3,
        'root_cause': root_cause,
        'class': classification['class'],
        'confidence': classification['confidence'],
        'actions': classification['actions'],
        'reasoning': reasoning_for(cluster, root_cause),
        'similar_incidents': classification['similar_incidents'],
        'method': classification['method'],
    })

rca_output = {
    'clusters_analyzed': len(results),
    'results': results,
}
OUT_PATH.parent.mkdir(parents=True, exist_ok=True)
OUT_PATH.write_text(json.dumps(rca_output, indent=2), encoding='utf-8')
print(json.dumps(rca_output, indent=2))

{
  "clusters_analyzed": 3,
  "results": [
    {
      "cluster_id": "c-001-000",
      "graph_top3": [
        [
          "payment-svc",
          0.77
        ],
        [
          "checkout-svc",
          0.47
        ],
        [
          "cart-svc",
          0.44
        ]
      ],
      "root_cause": "payment-svc",
      "class": "connection_pool_exhaustion",
      "confidence": 0.9,
      "actions": [
        "Rollback latest payment-svc deploy",
        "Scale payment-svc workers",
        "Temporarily raise connection pool limit"
      ],
      "reasoning": "payment-svc is ranked highest because it is downstream in the dependency graph, can explain upstream/victim alerts, and its alert appears early in the cluster time range.",
      "similar_incidents": [
        "INC-2026-05-10",
        "INC-2026-06-01",
        "INC-2026-03-22"
      ],
      "method": "graph+retrieval"
    },
    {
      "cluster_id": "c-001-001",
      "graph_top3": [
        [
          "recommende

In [6]:
validated = json.loads(OUT_PATH.read_text(encoding='utf-8'))
assert validated['clusters_analyzed'] == clusters_doc['output_clusters']
assert all('graph_top3' in row and row['graph_top3'] for row in validated['results'])
assert all('root_cause' in row and 'class' in row for row in validated['results'])
print('valid_json=True')
print('clusters_analyzed=', validated['clusters_analyzed'])
print('root_causes=', [row['root_cause'] for row in validated['results']])

valid_json=True
clusters_analyzed= 3
root_causes= ['payment-svc', 'recommender-svc', 'search-svc']
